# Cheminformatics basics with RDKit

RDKit parses molecules from a simple text notation (SMILES), computes their properties, and
compares them structurally. We look at three drugs - aspirin, ibuprofen, caffeine - compute
basic descriptors, then check that a structural similarity measure agrees with what a chemist
already knows: aspirin and ibuprofen are both NSAID-like carboxylic acids, so they should be
measurably more similar to each other than either is to caffeine (a purine alkaloid, a
completely different kind of molecule).

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

molecules = {
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
}

mols = {name: Chem.MolFromSmiles(smi) for name, smi in molecules.items()}
for name, mol in mols.items():
    print(f"{name:>10}  MW={Descriptors.MolWt(mol):6.1f}  LogP={Descriptors.MolLogP(mol):5.2f}")

## Structural similarity via fingerprints

A Morgan fingerprint encodes each atom's local neighborhood (out to a given radius) as a bit
vector. Comparing fingerprints with the Tanimoto coefficient gives a 0-1 similarity score -
this is the standard way cheminformatics tools do similarity search and clustering.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator, DataStructs

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = {name: gen.GetFingerprint(mol) for name, mol in mols.items()}

sim_ai = DataStructs.TanimotoSimilarity(fps["aspirin"], fps["ibuprofen"])
sim_ac = DataStructs.TanimotoSimilarity(fps["aspirin"], fps["caffeine"])
print(f"Tanimoto(aspirin, ibuprofen) = {sim_ai:.3f}")
print(f"Tanimoto(aspirin, caffeine)  = {sim_ac:.3f}")

assert sim_ai > sim_ac
print("\naspirin is more similar to ibuprofen than to caffeine, as expected")

## Draw the molecules

A quick visual sanity check - do the drawn structures actually look like what you'd expect?

In [ ]:
from rdkit.Chem import Draw

Draw.MolsToGridImage(list(mols.values()), legends=list(mols.keys()), molsPerRow=3, subImgSize=(250, 250))

## Next steps

- Add your own molecules by SMILES and see where they land in the similarity ranking.
- Compute more descriptors (`Descriptors.NumHDonors`, `Descriptors.TPSA`, ...) and check them
  against Lipinski's "rule of five" for drug-likeness.
- Try substructure search: `mol.HasSubstructMatch(Chem.MolFromSmarts("C(=O)O"))` to find which
  molecules contain a carboxylic acid group.